# 1. Working with Targets

In this tutorial, you will learn how to initialize the `Astrometrics` high-level interface, fetch targets from the catalog, and inspect the `Target` domain object.


## 1.1 Initializing Astrometrics

The `Astrometrics` object is your primary entry point. It provides access to the `targets`, `processing`, `visualization`, `stars`, and `moving_objects` registries.


In [1]:
from astrometricslib import Astrometrics

astrometrics = Astrometrics()
print("Astrometrics initialized successfully.")

Astrometrics initialized successfully.


## 1.2 Loading the Bundled Sample Data

The repository ships a small set of real M 13 images under `documentation/notebooks/astrometrics/sample_data/`, so this tutorial series runs without a telescope of your own. Its `README.md` records where those images came from.

There are two kinds of image in that folder:

- **Luminance frames** are taken through a clear filter that passes all visible light. They are ordinary pictures of the cluster, and they feed the imaging pipelines.
- **Spectroscopy frames** are taken through a grating, which spreads each star's light out into a spectrum. They feed the spectroscopy pipeline, and carry `SPEC` as their filter name.

The two kinds cannot share a target. Every frame on a target gets stacked together, and stacking a picture on top of a spectrum would produce nothing meaningful, so `astrometricslib` keeps them apart.

We therefore register two targets: **M 13** holds the Luminance frames for the imaging pipelines, and **M 13 Spectroscopy** holds the spectroscopy frames.


In [2]:
from documentation.notebooks.astrometrics.scripts.sample_data_staging import (
    drop_stale_sample_data_frames,
    stage_m13_sample_data,
)

sample_data_dir = stage_m13_sample_data()

imaging_target = astrometrics.targets.get("M 13") or astrometrics.targets.create("M 13")
spectroscopy_target = astrometrics.targets.get("M 13 Spectroscopy") or astrometrics.targets.create(
    "M 13 Spectroscopy"
)
drop_stale_sample_data_frames(imaging_target)
drop_stale_sample_data_frames(spectroscopy_target)

for path in sorted(sample_data_dir.glob("M_13_Light_Luminance_*.fits")):
    if not any(f.path == str(path) for f in imaging_target.frames):
        astrometrics.targets.add_frame(imaging_target, path=str(path), role="LIGHT")

for path in sorted(sample_data_dir.glob("M_13_Light_Spectroscopy_*.fits")):
    if not any(f.path == str(path) for f in spectroscopy_target.frames):
        astrometrics.targets.add_frame(spectroscopy_target, path=str(path), role="LIGHT")

astrometrics.targets.save()
print(f"M 13: {len(imaging_target.frames)} Luminance frames registered.")
print(f"M 13 Spectroscopy: {len(spectroscopy_target.frames)} Spectroscopy frames registered.")

M 13: 5 Luminance frames registered.
M 13 Spectroscopy: 5 Spectroscopy frames registered.


## 1.3 Fetching Targets

You can list all targets or get a specific target using the `astrometrics.targets` registry.


In [3]:
targets = astrometrics.targets.list()
print(f"Found {len(targets)} targets in the catalog.")

# Let's get a specific target, for example M 13
target = astrometrics.targets.get("M 13")
print(f"Fetched Target: {target.id}")

Found 2 targets in the catalog.
Fetched Target: M 13


## 1.4 Inspecting the Target Object

The `Target` object holds metadata and a list of `FrameRecord` objects representing the FITS files associated with this target.


In [4]:
print(f"Target ID: {target.id}")
print(f"Target Type: {target.image_type}")
print(f"Total Exposure Time: {target.exposure_sec} seconds")
print(f"Number of registered frames: {len(target.frames)}")

Target ID: M 13
Target Type: target_image
Total Exposure Time: 150.0 seconds
Number of registered frames: 5


## 1.5 Inspecting Frame Records

Let's look at the first few frame records attached to this target.


In [5]:
for i, frame in enumerate(target.frames[:3]):
    print(f"Frame {i + 1}:")
    print(f"  Role: {frame.role}")
    print(f"  Filter: {frame.filter}")
    print(f"  Exposure: {frame.exposure}s")
    print(f"  Path: {frame.path}")
    print("-" * 20)

Frame 1:
  Role: LIGHT
  Filter: Luminance
  Exposure: 30.0s
  Path: /home/user/astrometrics/libraryIndex/sample_data_working/M 13/M_13_Light_Luminance_019.fits
--------------------
Frame 2:
  Role: LIGHT
  Filter: Luminance
  Exposure: 30.0s
  Path: /home/user/astrometrics/libraryIndex/sample_data_working/M 13/M_13_Light_Luminance_020.fits
--------------------
Frame 3:
  Role: LIGHT
  Filter: Luminance
  Exposure: 30.0s
  Path: /home/user/astrometrics/libraryIndex/sample_data_working/M 13/M_13_Light_Luminance_021.fits
--------------------
